# DS2002 · Walmart Warm-Up SKU Mess

**Lab — 2026-10-16 · Fall 2026**  

---

## Lab 08 — Walmart Warm-Up (SKU Mess)

This lab uses the **real midterm data** in `../06-midterm-walmart/data/`. It's a warm-up so you hit the ground running on the project. On Kaggle/Colab, upload the data folder or adjust the path.

*If the file isn't found, the cell falls back to a small inline sample so the notebook still runs.*

In [8]:
import pandas as pd
PATH = '../06-midterm-walmart/data/product_catalog.csv'
try:
    cat = pd.read_csv(PATH)
except FileNotFoundError:
    from io import StringIO
    cat = pd.read_csv(StringIO('sku,product_name,category\nPT-12,pop-tarts strawberry,Breakfast Foods\nSKU#459812,Strawberry Pop-Tart,Breakfast\nPOPTART-STRAW,Pop-Tarts Strawberry,Breakfast'))
cat.head()

,sku,product_name,category
0,PT-12,pop-tarts strawberry,Breakfast Foods
1,SKU#459812,Strawberry Pop-Tart,Breakfast
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast


### Q1 — How many rows look like Pop-Tart products?

Filter on `product_name` containing `pop`, case-insensitively. Print the count and the matching rows.

In [9]:
mask = cat['product_name'].str.lower().str.contains('pop')
pop_rows = cat.loc[mask]

print("Count:", pop_rows.shape[0])
pop_rows


Count: 3


,sku,product_name,category
0,PT-12,pop-tarts strawberry,Breakfast Foods
1,SKU#459812,Strawberry Pop-Tart,Breakfast
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast


### Q2 — Which SKUs and names all mean *strawberry* Pop-Tart?

List them. Then note which Pop-Tart rows are **not** strawberry — those must stay separate, and saying so is part of the answer.

In [10]:
straw_mask = pop_rows['product_name'].str.lower().str.contains('straw')
straw = pop_rows.loc[straw_mask]
non_straw = pop_rows.loc[~straw_mask]

print("Strawberry Pop-Tart SKUs and names:")
print(straw[['sku', 'product_name']])

print("\nPop-Tart rows that are NOT strawberry (keep separate):")
print(non_straw[['sku', 'product_name']])


Strawberry Pop-Tart SKUs and names:
             sku          product_name
0          PT-12  pop-tarts strawberry
1     SKU#459812   Strawberry Pop-Tart
2  POPTART-STRAW  Pop-Tarts Strawberry

Pop-Tart rows that are NOT strawberry (keep separate):
Empty DataFrame
Columns: [sku, product_name]
Index: []


**Pop-Tart rows that are a different product:** All three Pop‑Tart rows refer to strawberry Pop‑Tarts, just with different SKUs and name formats.
There are no non‑strawberry Pop‑Tart rows in this sample, so nothing needs to stay separate.

### Q3 — Build the mapping

A dict from each variant SKU to one canonical SKU. Then apply it in a way that leaves unmapped SKUs as themselves rather than turning them into `NaN` — that single mistake silently deletes most of the catalog.

In [11]:
canonical = 'PT-12'

SKU_MAP = {
    'PT-12': canonical,
    'SKU#459812': canonical,
    'POPTART-STRAW': canonical
}
cat['canonical_sku'] = cat['sku'].map(SKU_MAP).fillna(cat['sku'])

cat[['sku', 'canonical_sku', 'product_name']]


,sku,canonical_sku,product_name
0,PT-12,PT-12,pop-tarts strawberry
1,SKU#459812,PT-12,Strawberry Pop-Tart
2,POPTART-STRAW,PT-12,Pop-Tarts Strawberry


### Q4 — Normalize `category`

Strip and title-case, then show the distinct values before and after with counts. Note any pair that is still two labels for one thing — `Breakfast` and `Breakfast Foods` are not the same string but they are the same category.

In [12]:
before = cat['category'].astype(str).unique()
print("Before:", before, "count =", len(before))

cat['category_norm'] = (
    cat['category']
        .astype(str)
        .str.strip()
        .str.title()
)

after = cat['category_norm'].unique()
print("\nAfter:", after, "count =", len(after))


Before: ['Breakfast Foods' 'Breakfast'] count = 2

After: ['Breakfast Foods' 'Breakfast'] count = 2


### Q5 — Find the fragmentation you have not spotted yet

Hand-scanning a catalog does not scale. **TODO:** build a candidate list programmatically — normalize each product name (lowercase, strip punctuation, sort the words) and group by that, then show any group holding more than one SKU.

This is the workflow from Monday's lecture, and it is how you will actually approach the full catalog on the midterm.

In [13]:
import re

def normalize_name(s):
    s = s.lower()
    s = re.sub(r'[^a-z0-9 ]', ' ', s)
    words = s.split()
    words = sorted(words)
    return " ".join(words)

cat['name_key'] = cat['product_name'].astype(str).apply(normalize_name)

groups = cat.groupby('name_key')

print("Candidate collisions (groups with >1 SKU):\n")
for key, grp in groups:
    if grp['sku'].nunique() > 1:
        print(f"Key: {key}")
        print(grp[['sku', 'product_name']])
        print()


Candidate collisions (groups with >1 SKU):

Key: pop strawberry tarts
             sku          product_name
0          PT-12  pop-tarts strawberry
2  POPTART-STRAW  Pop-Tarts Strawberry



### Q6 — Prove the mapping is safe

**TODO:** assert that applying your mapping produced no nulls in the canonical column and did not change the number of rows. Consolidation renames; it never adds or removes.

In [14]:
assert cat['canonical_sku'].isna().sum() == 0, "ERROR: mapping introduced nulls"

assert len(cat) == len(cat['canonical_sku']), "ERROR: row count changed"

print("Mapping is safe: no nulls, no row loss.")


Mapping is safe: no nulls, no row loss.


### Write-up

**a)** Name two product families beyond Pop-Tarts that appear fragmented, and how you would group them.

**b)** Where does consolidation stop being a data-cleaning task and become a business decision? Give one example from this catalog where you genuinely are not sure, and say who you would ask.

a) Beyond Pop-Tarts, the catalog shows fragmentation in the *Breakfast* vs *Breakfast Foods* category labels, which clearly refer to the same aisle but appear under different names. These should be grouped into one canonical category because the difference is only wording. A second fragmented family is the set of Strawberry Pop-Tart name variants, where the same product appears with different  word orders and punctuation. These should be consolidated into one canonical product name since they all describe the same term.

b) Consolidation stops being a pure cleaning task when two labels might represent different merchandising groupes even if they look similar. For example, I cannot be completely sure whether *Breakfast* and *Breakfast Foods* are intended to be identical or whether one is meant to be a narrower subset. At that point the data alone isn’t enough, and I would ask the category manager or merchandising lead responsible for the breakfast aisle to confirm whether those labels should be merged or kept separate.